# Lab 2 — Follow the Data

**Agentic AI Governance Practitioner** · Week 2

**Time:** about 25 minutes. **No Python knowledge needed.**

---

## What you're doing

You'll run **Microsoft Presidio** — a real, free, open-source PII scanner that people deploy at work — against two things:

1. SupportFlow's **customer conversations**
2. SupportFlow's **application logs**

Then you'll compare them.

> **Prefer not to run code?** Open `data/presidio_findings_prebuilt.csv` in the case packet instead. It contains these exact findings. **Your Section 2 will be graded identically.** The skill this week is *interpreting* the scan, not running it.

## How to run

**File → Save a copy in Drive**, then **Runtime → Run all**.


## Step 1 — Install

This takes about 2 minutes. Presidio downloads a language model.


In [ ]:
%%capture
!pip install -q presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_lg
!git clone -q https://github.com/francoisarthanas/agentic-gov-labs.git /content/labs 2>/dev/null || (cd /content/labs && git pull -q)


In [ ]:
import sys, json, glob, os
import pandas as pd
sys.path.insert(0, '/content/labs')
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()
print('✅ Presidio ready.')


In [ ]:
ENTITIES = ['PERSON','EMAIL_ADDRESS','PHONE_NUMBER','LOCATION','CREDIT_CARD',
            'US_SSN','IP_ADDRESS','DATE_TIME','URL','US_BANK_NUMBER']

def scan(text, source, line_no, field):
    if not text or not isinstance(text, str):
        return []
    try:
        res = analyzer.analyze(text=text, entities=ENTITIES, language='en')
    except Exception:
        return []
    return [{'source': source, 'line': line_no, 'field': field,
             'entity_type': r.entity_type,
             'detected_text': text[r.start:r.end][:80],
             'confidence': round(r.score, 3)} for r in res]

print('✅ Scanner function defined.')


## Step 2 — Scan the customer conversations

Three real SupportFlow transcripts. **Before you run this: how much personal data do you expect to find?**


In [ ]:
rows = []
for path in sorted(glob.glob('/content/labs/data/transcripts/*.md')):
    name = os.path.basename(path)
    for i, line in enumerate(open(path), 1):
        rows += scan(line.strip(), name, i, 'transcript')

transcripts_df = pd.DataFrame(rows)
print(f'Findings in transcripts: {len(transcripts_df)}')
transcripts_df.groupby('entity_type').size().sort_values(ascending=False)


### ✋ Stop here for a moment

**Write down your prediction before running the next cell.**

You just scanned the conversations customers actually had. Now you're going to scan the application logs from those same conversations.

**How much PII do you expect in the logs, compared to the transcripts?**

- About the same?
- Somewhat more?
- Somewhat less?

Commit to an answer. Then run the next cell.


## Step 3 — Scan the logs


In [ ]:
rows = []
with open('/content/labs/data/logs_sample.jsonl') as f:
    for i, line in enumerate(f, 1):
        rec = json.loads(line)
        rows += scan(json.dumps(rec), 'logs_sample.jsonl', i, 'full_log_record')

logs_df = pd.DataFrame(rows)
print(f'Findings in logs: {len(logs_df)}')
logs_df.groupby('entity_type').size().sort_values(ascending=False)


## Step 4 — Compare


In [ ]:
t, l = len(transcripts_df), len(logs_df)
print(f'Transcripts : {t:>6}')
print(f'Logs        : {l:>6}')
print(f'Ratio       : {l/max(t,1):>6.1f}x more PII in the logs')
print()
print('Was that your prediction?')


### Why?

Look at one of the log lines that produced the most findings.


In [ ]:
with open('/content/labs/data/logs_sample.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        if rec.get('event') == 'model_request':
            print(json.dumps(rec, indent=2)[:2200])
            break


**That is a single log line.**

It contains the entire context window sent to the model provider — including the full `crm_lookup` response: name, email, phone, shipping address, payment last-4, lifetime value, refund history.

**The customer never saw most of those fields. The log has all of them.**

So: **your log retention is now your PII retention.** And your log retention was probably set by whoever configured the observability stack for debugging convenience, years ago, without a privacy review.


## Step 5 — Question the tool

Presidio is a tool, not an oracle. It gets things wrong in **both** directions.

Run this and look carefully.


In [ ]:
checks = ['1721000000', 'card ending 8802', 'ORD-2026-4417', 'C-1041',
          'RF-2026-88012', '$4,215.50', 'marcus.webb@example.com']

for c in checks:
    res = analyzer.analyze(text=c, language='en')
    found = [(r.entity_type, round(r.score,2)) for r in res]
    print(f'{c:26s} -> {found if found else "❌ NOT FLAGGED"}')


### What you should notice

**False positives** — the scanner flags things that aren't what it says:
- `1721000000` is a Unix timestamp, flagged as a bank account number. Every single one of the ~500 `US_BANK_NUMBER` hits is a timestamp.
- `card ending 8802` is payment data, flagged as a *date*.

**Misses** — the scanner does not flag things that matter:
- `C-1041` is a **direct customer identifier**. Not flagged.
- `ORD-2026-4417` links straight to a person. Not flagged.

> **The lesson:** out-of-the-box scanners find *formats*, not *your* identifiers. If you take the raw number to your CISO, you'll be wrong in both directions at once.
>
> This is why your **Result** column has to say what you actually reviewed.


## Step 6 — Export your findings


In [ ]:
all_df = pd.concat([transcripts_df, logs_df], ignore_index=True)
all_df.to_csv('my_presidio_findings.csv', index=False)
print(f'✅ Exported {len(all_df)} findings to my_presidio_findings.csv')
print('   Download it from the file browser on the left,')
print('   then attach it to Evidence Pack §2 in VerifyWise.')


---

## ✅ Done — now write it up

Three questions for your Section 2:

1. **Which of the nine data surfaces did this scan actually cover?** (Hint: fewer than nine. Which ones did you *not* look at, and why not?)
2. **Who owns log retention at Northwind, and what is it?** If you don't know, that's an open item with a named owner.
3. **Write one evidence row with all six columns** — Claim, Control, Test, Result, Owner, Trigger. It is fine — better, even — for the Result to be a **FAIL**.

> A Section 2 full of passes, for a system nobody has hardened, is a story. A recorded failure with a named owner is evidence.
